# InsureLLM — RAG Question Answering

A retrieval-augmented question-answering application for InsureLLM.

The application retrieves relevant passages from a persistent **ChromaDB** vector store, then uses a **LangChain** chat model to generate a grounded response. A **Gradio** chat interface provides the user-facing application.

## Pipeline

1. Load the existing ChromaDB collection.
2. Embed incoming questions with `all-MiniLM-L6-v2`.
3. Retrieve the most relevant document chunks.
4. Pass the retrieved context to the chat model.
5. Return the generated answer through Gradio.

## Requirements

- A populated `vector_db/` directory created by the document-ingestion notebook.
- An OpenAI API key.
- The Python packages listed below.

Set `OPENAI_API_KEY` and, optionally, `OPENAI_MODEL` in your environment or in a local `.env` file.

## 1. Dependencies

In [ ]:
import os
from pathlib import Path

import gradio as gr
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI

## 2. Configuration

In [ ]:
load_dotenv()

DB_PATH = Path("vector_db")
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
LLM_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError(
        "OPENAI_API_KEY is not set. Add it to your environment or local .env file."
    )

if not DB_PATH.exists():
    raise FileNotFoundError(
        f"Vector store not found at '{DB_PATH}'. Run the document-ingestion notebook first."
    )

## 3. Load the vector store

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL
)

vectorstore = Chroma(
    persist_directory=str(DB_PATH),
    embedding_function=embeddings,
)

## 4. Initialize retrieval and the language model

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

llm = ChatOpenAI(
    model=LLM_MODEL,
    temperature=0,
)

## 5. Define the answer prompt

In [ ]:
SYSTEM_PROMPT_TEMPLATE = """You are a knowledgeable and helpful assistant for InsureLLM.

Answer the user's question using the retrieved context below.
If the context does not contain enough information to answer, say that you
do not have enough information. Do not invent facts.

Retrieved context:
{context}
"""

## 6. Implement the RAG chain

In [ ]:
def answer_question(question: str, history: list) -> str:
    """Retrieve relevant documents and generate a context-grounded answer.

    Args:
        question: The user's current question.
        history: Gradio chat history (reserved for conversational extensions).

    Returns:
        The generated answer as a string.
    """
    documents = retriever.invoke(question)
    context = "\\n\\n".join(document.page_content for document in documents)

    system_message = SystemMessage(
        content=SYSTEM_PROMPT_TEMPLATE.format(context=context)
    )
    user_message = HumanMessage(content=question)

    response = llm.invoke([system_message, user_message])
    return response.content

## 7. Launch the chat interface

In [ ]:
chatbot = gr.ChatInterface(
    fn=answer_question,
    title="InsureLLM Assistant",
    description="Ask questions about InsureLLM's knowledge base.",
)

chatbot.launch()

## Notes

- The notebook expects the `vector_db/` directory to be generated by the document-ingestion workflow.
- The embedding model used to query the collection must match the model used when the documents were indexed.
- Keep API keys in environment variables or a local `.env` file. Do not commit secrets to GitHub.
- The Gradio interface is launched locally by default.